In [14]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import wandb
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import efficientnet.keras as efn 
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import lightning as L
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.metrics import classification_report, confusion_matrix

# YOLO könyvtár beállítása
PROJECT_ROOT = Path('/work')
YOLOV5_DIR = PROJECT_ROOT / 'external' / 'yolov5'
if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))
from models.yolo import Model
from utils.general import non_max_suppression

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Használt eszköz: {DEVICE}")

# --- KONFIGURÁCIÓ ---
CLASSIFICATION_DATASET_ROOT = Path('/work/data/classification_dataset')
TEST_SPLIT = 'test'
COCO_FILENAME = '_annotations.coco.json'

# Kérlek, ellenőrizd, hogy a lokális futtatásnál ezek jók-e (ha nem Dockerben vagy)
YOLO_WEIGHTS = Path('/work/final_models/detection.pt') 
CLF_WEIGHTS = Path('/work/final_models/classification.pt')

CARIES_CATEGORY_ID = 44 # A JSON-ben ez a szuvas fog kódja

wandb.init(
    project="tooth-e2e-evaluation",
    name="pipeline-eval",
    job_type="evaluation"
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Használt eszköz: cuda


In [15]:
class LitYOLOv5(torch.nn.Module):
    def __init__(self, image_size=640, num_classes=32):
        super().__init__()
        yolo_cfg = YOLOV5_DIR / 'models' / 'yolov5s.yaml'
        self.model = Model(yolo_cfg, ch=3, nc=num_classes).float()

class LitToothClassifier(L.LightningModule):
    def __init__(self, num_classes=80):
        super().__init__()

        self.model = efficientnet_b0(
            weights=EfficientNet_B0_Weights.DEFAULT
        )

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = torch.nn.Linear(
            in_features,
            num_classes
        )

    def forward(self, x):
        return self.model(x)

def compute_zoom_crop_box(width, height, zoom=0.90, cx=0.50, cy=0.666):
    crop_w, crop_h = int(width * zoom), int(height * zoom)
    x1, y1 = max(0, int(width * cx) - crop_w // 2), max(0, int(height * cy) - crop_h // 2)
    return int(x1), int(y1), min(width, x1 + crop_w), min(height, y1 + crop_h)

def prepare_image_for_yolo(orig_img, img_size=640):
    orig_w, orig_h = orig_img.size
    zx1, zy1, zx2, zy2 = compute_zoom_crop_box(orig_w, orig_h)
    inf_img = orig_img.crop((zx1, zy1, zx2, zy2))
    scale = img_size / max(inf_img.size)
    new_w, new_h = int(inf_img.width * scale), int(inf_img.height * scale)
    padded = TF.pad(inf_img.resize((new_w, new_h), Image.Resampling.BILINEAR), (0, 0, img_size - new_w, img_size - new_h), fill=0)
    return TF.to_tensor(padded), {'orig_w': orig_w, 'orig_h': orig_h, 'zoom_box': (zx1, zy1, zx2, zy2), 'scale': scale, 'new_w': new_w, 'new_h': new_h}

def yolo_boxes_to_original_xyxy(boxes, meta):
    if len(boxes) == 0: return boxes.clone().float()
    boxes = boxes.clone().float()
    scale = float(meta['scale'])
    zx1, zy1, zx2, zy2 = meta['zoom_box']
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, meta['new_w']) / scale + zx1
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, meta['new_h']) / scale + zy1
    return boxes

def box_iou_xyxy(boxes1, boxes2):
    if len(boxes1) == 0 or len(boxes2) == 0: return torch.zeros((len(boxes1), len(boxes2)), device=boxes1.device)
    x1 = torch.max(boxes1[:, None, 0], boxes2[None, :, 0])
    y1 = torch.max(boxes1[:, None, 1], boxes2[None, :, 1])
    x2 = torch.min(boxes1[:, None, 2], boxes2[None, :, 2])
    y2 = torch.min(boxes1[:, None, 3], boxes2[None, :, 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    return inter / (area1[:, None] + area2[None, :] - inter).clamp(min=1e-6)

def apply_two_stage_heuristic(boxes, scores, labels, iou_threshold=0.75):
    keep_indices = [int(torch.where(labels == lbl)[0][torch.argmax(scores[torch.where(labels == lbl)[0]])]) for lbl in torch.unique(labels)]
    keep_indices = sorted(keep_indices)
    boxes, scores, labels = boxes[keep_indices], scores[keep_indices], labels[keep_indices]
    
    if len(boxes) <= 1: return boxes, scores, labels
    order = torch.argsort(scores, descending=True)
    final_keep = []
    for idx in order.tolist():
        if not final_keep:
            final_keep.append(idx)
            continue
        if torch.max(box_iou_xyxy(boxes[idx].unsqueeze(0), boxes[final_keep])[0]) <= iou_threshold:
            final_keep.append(idx)
    final_keep = sorted(final_keep)
    return boxes[final_keep], scores[final_keep], labels[final_keep]

In [17]:
print("Modellek betöltése...")
yolo_model = LitYOLOv5(image_size=640, num_classes=32)
yolo_model.model.load_state_dict(torch.load(YOLO_WEIGHTS, map_location='cpu'), strict=False)
yolo_model.to(DEVICE).eval()

classifier_model = torch.load(CLF_WEIGHTS, map_location=DEVICE) 
classifier_model.to(DEVICE).eval()

efficientnet_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def resolve_image_path(split_dir: Path, file_name: str) -> Path:
    """Útvonal feloldása a mappaszerkezetben"""
    direct = split_dir / file_name
    if direct.exists():
        return direct
    matches = list(split_dir.rglob(Path(file_name).name))
    if matches:
        return matches[0]
    return None

def load_test_records(dataset_root: Path, split: str):
    """Rekordok felépítése a COCO JSON alapján, 'cate' szűréssel"""
    ann_path = dataset_root / split / COCO_FILENAME
    split_dir = dataset_root / split
    
    with open(ann_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)
        
    img_id_to_info = {img['id']: img for img in coco['images']}
    anns_by_image_id = {}
    for ann in coco['annotations']:
        img_id = ann['image_id']
        if img_id not in anns_by_image_id:
            anns_by_image_id[img_id] = []
        anns_by_image_id[img_id].append(ann)
        
    records = []
    for img_id, img_info in img_id_to_info.items():
        raw_file_name = img_info['file_name']
        
        # 'cate' szűrés alkalmazása
        if not Path(raw_file_name).name.lower().startswith('cate'):
            continue
            
        img_path = resolve_image_path(split_dir, raw_file_name)
        if img_path is None:
            continue
            
        records.append({
            'image_id': img_id,
            'image_path': img_path,
            'annotations': anns_by_image_id.get(img_id, [])
        })
    return records

print(f"Adatok betöltése innen: {CLASSIFICATION_DATASET_ROOT / TEST_SPLIT}")
test_records = load_test_records(CLASSIFICATION_DATASET_ROOT, TEST_SPLIT)
print(f"Összesen {len(test_records)} 'cate' típusú teszt kép betöltve.")

Overriding model.yaml nc=80 with nc=32

                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  6                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  7                -1  1   1180672  models.common.Conv                      [256, 512, 3, 2]            

Modellek betöltése...


/tmp/ipykernel_15471/3402645180.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  yolo_model.model.load_state_dict(torch.load(YOLO_WEIGHTS, map_location='cpu'), strict=Fal

Adatok betöltése innen: /work/data/classification_dataset/test
Összesen 265 'cate' típusú teszt kép betöltve.


In [18]:
map_metric = MeanAveragePrecision(box_format="xyxy", class_metrics=True)
metric_preds = []
metric_targets = []

matched_y_true = []
matched_y_pred = []
matched_y_probs = []

print("E2E kiértékelés indítása a teszt adatokon...")
for record in tqdm(test_records, desc="Képek feldolgozása"):
    img_path = record['image_path']
    try:
        orig_img = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"Hiba a kép beolvasásakor: {img_path} - {e}")
        continue
    
    # GT adatok előkészítése a betöltött rekordból
    gt_boxes = []
    gt_labels = []
    for ann in record['annotations']:
        bbox = ann['bbox'] # [x, y, w, h]
        gt_boxes.append([bbox[0], bbox[1], bbox[0]+bbox[2], bbox[1]+bbox[3]])
        gt_labels.append(1 if ann['category_id'] == CARIES_CATEGORY_ID else 0)
    
    target_dict = {
        "boxes": torch.tensor(gt_boxes, dtype=torch.float32, device=DEVICE).reshape(-1, 4),
        "labels": torch.tensor(gt_labels, dtype=torch.long, device=DEVICE)
    }
    metric_targets.append(target_dict)
    
    # Inferencia (YOLO + ResNet) 
    img_tensor, prep_meta = prepare_image_for_yolo(orig_img)
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        yolo_out = yolo_model.model(img_tensor)[0]
        nms_preds = non_max_suppression(yolo_out, conf_thres=0.29, iou_thres=0.45, max_det=200)[0]
    
    pred_boxes_list, pred_scores_list, pred_labels_list = [], [], []
    
    if nms_preds is not None and len(nms_preds) > 0:
        boxes, scores, labels = nms_preds[:, :4].cpu(), nms_preds[:, 4].cpu(), nms_preds[:, 5].cpu()
        filtered_boxes, _, _ = apply_two_stage_heuristic(boxes, scores, labels)
        pred_boxes_original = yolo_boxes_to_original_xyxy(filtered_boxes, prep_meta)
        
        for box in pred_boxes_original:
            x1, y1, x2, y2 = [int(v) for v in box.tolist()]
            # Klasszifikáció futtatása a kivágáson
            tooth_crop = orig_img.crop((max(0,x1), max(0,y1), min(orig_img.width,x2), min(orig_img.height,y2)))
            clf_input = efficientnet_transform(tooth_crop).unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                probs = torch.softmax(classifier_model(clf_input), dim=1)[0]
                caries_prob = probs[1].item()
            
            pred_class = 1 if caries_prob > 0.5 else 0
            pred_boxes_list.append([x1, y1, x2, y2])
            pred_scores_list.append(caries_prob if pred_class == 1 else probs[0].item())
            pred_labels_list.append(pred_class)

    # Metrika frissítése a MeanAveragePrecision-höz
    pred_dict = {
        "boxes": torch.tensor(pred_boxes_list, dtype=torch.float32, device=DEVICE).reshape(-1, 4),
        "scores": torch.tensor(pred_scores_list, dtype=torch.float32, device=DEVICE),
        "labels": torch.tensor(pred_labels_list, dtype=torch.long, device=DEVICE)
    }
    metric_preds.append(pred_dict)
    
    # Confusion matrix párosítás (IoU > 0.45 alapján)
    if len(gt_boxes) > 0 and len(pred_boxes_list) > 0:
        ious = box_iou_xyxy(target_dict["boxes"], pred_dict["boxes"])
        for gt_idx in range(len(gt_boxes)):
            max_iou, p_idx = torch.max(ious[gt_idx], dim=0)
            if max_iou.item() > 0.45:
                matched_y_true.append(gt_labels[gt_idx])
                matched_y_pred.append(pred_labels_list[p_idx.item()])
                
                # Valószínűségek a ROC és PR görbéhez
                p1 = pred_dict["scores"][p_idx].item() if pred_labels_list[p_idx.item()] == 1 else 1 - pred_dict["scores"][p_idx].item()
                matched_y_probs.append([1 - p1, p1])

print("Kiértékelési ciklus befejeződött.")

E2E kiértékelés indítása a teszt adatokon...


Képek feldolgozása: 100%|██████████| 265/265 [01:30<00:00,  2.93it/s]

Kiértékelési ciklus befejeződött.


In [19]:
print("E2E mAP számolása...")
map_metric.update(metric_preds, metric_targets)
results = map_metric.compute()

print(f"--- E2E DETECTION METRICS ---")
print(f"mAP @ IoU 50:95: {results['map']:.4f}")
print(f"mAP @ IoU 50:    {results['map_50']:.4f}")
print(f"mAR @ MaxDets:   {results['mar_100']:.4f}")
print(f"mAP (Healthy=0): {results['map_per_class'][0]:.4f}")
if len(results['map_per_class']) > 1:
    print(f"mAP (Caries=1):  {results['map_per_class'][1]:.4f}")

wandb.log({
    "e2e/map_50_95": results['map'].item(),
    "e2e/map_50": results['map_50'].item(),
    "e2e/mar_100": results['mar_100'].item(),
})

# --- KLASSZIFIKÁCIÓS METRIKÁK ---
print("\n--- CLASSIFICATION METRICS (Matched Teeth Only) ---")
if len(matched_y_true) > 0:
    print(classification_report(matched_y_true, matched_y_pred, target_names=['Healthy', 'Caries']))
    
    # Logolás WandB-be
    wandb.log({
        "eval/interactive_cm": wandb.plot.confusion_matrix(
            probs=None, 
            y_true=matched_y_true, 
            preds=matched_y_pred, 
            class_names=['Healthy', 'Caries']
        ),
        "eval/interactive_roc": wandb.plot.roc_curve(
            matched_y_true, 
            np.array(matched_y_probs), 
            labels=['Healthy', 'Caries']
        ),
        "eval/interactive_pr": wandb.plot.pr_curve(
            matched_y_true, 
            np.array(matched_y_probs), 
            labels=['Healthy', 'Caries']
        )
    })
else:
    print("Nem történt sikeres bounding box párosítás.")

wandb.finish()
print("Kész! Nézd meg a WandB dashboardot az ábrákért.")

E2E mAP számolása...
--- E2E DETECTION METRICS ---
mAP @ IoU 50:95: 0.2075
mAP @ IoU 50:    0.3897
mAR @ MaxDets:   0.3415
mAP (Healthy=0): 0.3664
mAP (Caries=1):  0.0487

--- CLASSIFICATION METRICS (Matched Teeth Only) ---
              precision    recall  f1-score   support

     Healthy       0.98      0.93      0.96      6143
      Caries       0.18      0.51      0.27       185

    accuracy                           0.92      6328
   macro avg       0.58      0.72      0.61      6328
weighted avg       0.96      0.92      0.94      6328



wandb: ERROR The nbformat package was not found. It is required to save notebook history.


e2e/map_50,▁
e2e/map_50_95,▁
e2e/mar_100,▁
e2e/map_50,0.38969
e2e/map_50_95,0.20751
e2e/mar_100,0.3415


Kész! Nézd meg a WandB dashboardot az ábrákért.
